# 应用案例: 评估闸门的防洪效果

## 目的
水力模型的核心应用之一是设计和评估水工建筑物（如水库、闸门、堤防）对洪水的影响。通过模拟，我们可以在工程实施前，定量地分析这些建筑物能否有效地削减下游的洪峰流量，从而为防洪决策提供科学依据。

本教程将展示如何使用`preissmann_model`来模拟和评估一个防洪闸门的效能。

此笔记本将：
1.  设定一个基础的河道模型和一个“设计洪水”过程线。
2.  模拟**情景一（无闸门）**，得到自然状态下的下游洪水过程。
3.  模拟**情景二（有闸门）**，在河道中加入一个闸门，并再次模拟洪水过程。
4.  通过对比两个情景的下游流量过程线，来量化闸门对洪峰的削减效果。

## 1. 环境设置和模型准备

我们首先建立一个用于测试的河道，并定义一个上游的入流洪水过程。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.model import HydraulicModel
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection
from preissmann_model.structures import Gate

# --- 定义一个辅助函数来运行模型 ---
def run_scenario(model, q_inflow):
    num_steps = len(q_inflow)
    outflows = []
    for i in range(num_steps):
        model.step({'Q_inflow': q_inflow[i]}, model.dt)
        outflows.append(model.get_outflow())
    return np.array(outflows)

# --- 定义共享的河道几何和洪水输入 ---
num_nodes = 21
reach_geom = RiverReach(
    cross_sections=[RectangularCrossSection(width=20) for _ in range(num_nodes)],
    lengths=np.full(num_nodes - 1, 500.0),
    slope=0.001,
    manning_n=0.03
)
# 设计洪水过程线
design_flood = np.array([10, 15, 25, 50, 100, 150, 120, 90, 60, 40, 30, 20, 15] + [10]*12)
num_steps = len(design_flood)

## 2. 运行情景一: 无控（自然）状态

我们首先运行一个没有闸门的基准情景，以了解洪水在自然河道中的传播过程。

In [ ]:
print("运行情景一 (无闸门)...")
model_no_gate = HydraulicModel(name="no_gate", reach=reach_geom, dt=60.0, downstream_level=3.0)
model_no_gate.Q[:] = design_flood[0]
model_no_gate.Z[:] = 3.5
outflow_no_gate = run_scenario(model_no_gate, design_flood)
print("运行完毕。")

## 3. 运行情景二: 有控（带闸门）状态

现在，我们在河道的中点（节点10）加入一个闸门。为了模拟防洪操作，我们将闸门开度设为一个较小的值（例如1.0米），以限制下泄流量。

In [ ]:
print("\n运行情景二 (有闸门)...")
gate = Gate(name="flood_gate", node_index=10, opening_height=1.0, width=20)
model_with_gate = HydraulicModel(name="with_gate", reach=reach_geom, dt=60.0, downstream_level=3.0, structures=[gate])
model_with_gate.Q[:] = design_flood[0]
model_with_gate.Z[:] = 3.5
outflow_with_gate = run_scenario(model_with_gate, design_flood)
print("运行完毕。")

## 4. 结果对比与可视化

我们将两个情景的下游出口流量过程线进行对比。

In [ ]:
timesteps = np.arange(num_steps) * 60 / 3600 # hours
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(timesteps, design_flood, 'k--', label='Inflow Flood Hydrograph')
ax.plot(timesteps, outflow_no_gate, 'b-', label='Outflow (Without Gate)')
ax.plot(timesteps, outflow_with_gate, 'r-s', markersize=4, label='Outflow (With Gate)')

ax.set_title('Evaluating Flood Control Effect of a Gate')
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Discharge (m³/s)')
ax.legend()
ax.grid(True)
plt.show()

# --- 量化对比 ---
peak_natural = np.max(outflow_no_gate)
peak_controlled = np.max(outflow_with_gate)
peak_reduction = peak_natural - peak_controlled
reduction_percent = (peak_reduction / peak_natural) * 100

print(f"自然状态下洪峰流量: {peak_natural:.2f} m³/s")
print(f"闸门控制后洪峰流量: {peak_controlled:.2f} m³/s")
print(f"洪峰削减量: {peak_reduction:.2f} m³/s ({reduction_percent:.1f}%) ")

## 5. 分析上游水位

削减下游洪峰的代价是抬高了闸门上游的水位。我们也可以对比两个情景下，闸门上游（节点9）的水位变化。

In [ ]:
# 需要重新运行以获取水位历史
z9_no_gate = []
z9_with_gate = []
model_no_gate.Q[:] = design_flood[0]; model_no_gate.Z[:] = 3.5
model_with_gate.Q[:] = design_flood[0]; model_with_gate.Z[:] = 3.5

for i in range(num_steps):
    model_no_gate.step({'Q_inflow': design_flood[i]}, model_no_gate.dt)
    z9_no_gate.append(model_no_gate.Z[9])
    model_with_gate.step({'Q_inflow': design_flood[i]}, model_with_gate.dt)
    z9_with_gate.append(model_with_gate.Z[9])

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(timesteps, z9_no_gate, 'b-', label='Upstream Water Level (Without Gate)')
ax.plot(timesteps, z9_with_gate, 'r-s', markersize=4, label='Upstream Water Level (With Gate)')
ax.set_title('Upstream Water Level at Node 9')
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Water Elevation (m)')
ax.legend()
ax.grid(True)
plt.show()

## 6. 结论

通过这个案例，我们验证了：
1.  **闸门的防洪作用**: 在下游，闸门成功地将洪峰流量削减了约 **{reduction_percent:.1f}%**，起到了显著的防洪作用。
2.  **闸门的壅水作用**: 在上游，为了实现对流量的控制，闸门将水拦蓄起来，导致其上游水位显著升高。在实际工程中，需要确保这个抬高的水位不会淹没上游的保护对象。

这个工作流展示了如何使用模型来评估水工建筑物的利（下游防洪）弊（上游壅水），从而为工程设计和调度方案的制定提供决策支持。